# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/k-karimli/FlyRankWeek1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

#### Unit of analysis

One row represents the daily performance of one content item for one client on one report date.

Table:
- fact_content_daily_performance

Time window:
- March 2026

Prediction goal:
- Analyze ranking signals using search performance metrics.

Excluded:
- Future information and leakage-prone fields.

In [5]:
!pip -q install duckdb huggingface_hub

from huggingface_hub import login, snapshot_download
from google.colab import userdata

login(userdata.get("HF_TOKEN"))

dataset_path = snapshot_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset"
)

import duckdb
from pathlib import Path

con = duckdb.connect()

base = Path(dataset_path)

march_file = base / "fact_content_daily_performance" / "month=2026-03" / "data_0.parquet"

con.sql(f"""
CREATE OR REPLACE VIEW march AS
SELECT *
FROM read_parquet('{march_file}')
""")

print("March view created!")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 24 files:   0%|          | 0/24 [00:00<?, ?it/s]

March view created!


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_sessions
- ga4_engaged_sessions

### Context
- report_date
- client_hash_id
- content_hash_id

### Label
No supervised label is used in this notebook.

### Excluded
Future information and leakage-prone columns are excluded.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql("""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_sessions,
    ga4_engaged_sessions
FROM march
LIMIT 10
""")

┌─────────────┬─────────────────────────┬──────────────────────────┬─────────────────┬────────────┬───────────────────┬──────────────┬──────────────────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ gsc_impressions │ gsc_clicks │ gsc_avg_position  │ ga4_sessions │ ga4_engaged_sessions │
│    date     │         varchar         │         varchar          │      int64      │   int64    │      double       │    int64     │        int64         │
├─────────────┼─────────────────────────┼──────────────────────────┼─────────────────┼────────────┼───────────────────┼──────────────┼──────────────────────┤
│ 2026-03-01  │ client_73cda7b4e4f265ea │ content_b7e512995f79d5a6 │              20 │          0 │              3.35 │         NULL │                 NULL │
│ 2026-03-01  │ client_73cda7b4e4f265ea │ content_05597932fe4da067 │               1 │          0 │               0.0 │         NULL │                 NULL │
│ 2026-03-01  │ client_73cda7b4e4f265ea │ content_7a

## 3. Verify it with queries (grain, counts, missing values, windows)

Verification Query 1
- Count rows.

Verification Query 2
- Check date range.

Verification Query 3
- Check rows where gsc_data_available IS TRUE.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Row count")
con.sql("""
SELECT COUNT(*) AS total_rows
FROM march
""")

print("Date range")
con.sql("""
SELECT
MIN(report_date) AS start_date,
MAX(report_date) AS end_date
FROM march
""")

print("Availability")
con.sql("""
SELECT COUNT(*) AS available_rows
FROM march
WHERE gsc_data_available IS TRUE
""")

Row count
Date range
Availability


┌────────────────┐
│ available_rows │
│     int64      │
├────────────────┤
│        3611061 │
└────────────────┘

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Limitations

- The dataset is observational.
- It cannot prove causality.
- Client histories are unbalanced.
- Early rows may contain only GSC data.
- This notebook analyzes only March 2026.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql("""
SELECT
COUNT(*) AS total_rows,
COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_rows
FROM march
""")

┌────────────┬──────────┐
│ total_rows │ ga4_rows │
│   int64    │  int64   │
├────────────┼──────────┤
│    9841378 │   413966 │
└────────────┴──────────┘

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.